# YOLO26 Custom Training Workflow

---

This notebook keeps the training path compact: centralized configs, managed directories, a Roboflow download step, reusable training profiles, and a simple DAG that shows the happy path plus the optional scale-up branch.

### Before you start

Let's make sure we have access to a GPU before training. We can use `nvidia-smi` to check that.

In [19]:
import shutil
import subprocess

def check_gpu_access() -> None:
    nvidia_smi = shutil.which("nvidia-smi")
    if nvidia_smi is None:
        raise RuntimeError("nvidia-smi not found on PATH. Install NVIDIA drivers or use a GPU-enabled environment.")

    gpu_probe = subprocess.run([nvidia_smi], capture_output=True, text=True)
    if gpu_probe.returncode != 0:
        raise RuntimeError(f"nvidia-smi failed: {gpu_probe.stderr.strip() or 'unknown error'}")

    print(gpu_probe.stdout)

check_gpu_access()

Fri May 15 15:59:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.61                 Driver Version: 572.61         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090      WDDM  |   00000000:41:00.0  On |                  Off |
|  0%   48C    P8             33W /  450W |    2596MiB /  24564MiB |      5%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**Note:** The notebook now manages its own workspace layout from the config block above, so datasets, results, models, logs, and artifacts all stay under one predictable root.

In [ ]:
from pathlib import Path

WORKSPACE_ROOT = Path.cwd().resolve()
NOTEBOOKS_ROOT = WORKSPACE_ROOT if WORKSPACE_ROOT.name == "notebooks" else WORKSPACE_ROOT / "notebooks"
NOTEBOOK_DIR = NOTEBOOKS_ROOT / "test-setup-02-train-hparams"

PIPELINE_DIRS = {
    "datasets": NOTEBOOK_DIR / "datasets",
    "results": NOTEBOOK_DIR / "results",
    "models": NOTEBOOK_DIR / "models",
    "logs": NOTEBOOK_DIR / "logs",
    "artifacts": NOTEBOOK_DIR / "artifacts",
}

for directory in PIPELINE_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "basketball-player-detection-3-14"
ROBOFLOW_WORKSPACE = "roboflow-jvuqo"
ROBOFLOW_PROJECT = "basketball-player-detection-3-ycjdo"
ROBOFLOW_VERSION = 14
EXPORT_FORMAT = "yolo26"
WEIGHTS_NAME = "yolo26n.pt"

DATASET_DIR = PIPELINE_DIRS["datasets"] / DATASET_NAME
TRAIN_PROJECT_DIR = PIPELINE_DIRS["results"] / "yolo26" / DATASET_NAME
MODEL_OUTPUT_DIR = PIPELINE_DIRS["models"] / "yolo26" / DATASET_NAME
LOG_DIR = PIPELINE_DIRS["logs"] / "yolo26" / DATASET_NAME

TRAIN_WORKERS = 0

TRAINING_PROFILES = {
    "smoke": {
        "epochs": 1,
        "imgsz": 320,
        "batch": 1,
        "device": 0,
        "workers": TRAIN_WORKERS,
        "cache": False,
        "pretrained": True,
        "optimizer": "SGD",
        "lr0": 0.001,
        "cos_lr": False,
        "patience": 1,
        "fraction": 0.01,
        "mosaic": 0.0,
        "mixup": 0.0,
        "fliplr": 0.0,
        "close_mosaic": 1,
        "plots": False,
        "amp": True,
        "verbose": True,
        "save": False,
        "val": False,
    },
    "scale_up": {
        "epochs": 3,
        "imgsz": 480,
        "batch": 1,
        "device": 0,
        "workers": TRAIN_WORKERS,
        "cache": False,
        "pretrained": True,
        "optimizer": "AdamW",
        "lr0": 0.002,
        "cos_lr": True,
        "patience": 2,
        "fraction": 0.05,
        "mosaic": 0.25,
        "mixup": 0.0,
        "fliplr": 0.1,
        "close_mosaic": 1,
        "plots": True,
        "amp": True,
        "verbose": True,
        "save": True,
        "val": False,
    },
}

BASE_TRAIN_ARGS = {
    "project": str(TRAIN_PROJECT_DIR),
    "exist_ok": True,
}


def build_train_args(profile_name: str, data_yaml_path: Path) -> dict:
    profile = dict(TRAINING_PROFILES[profile_name])
    return {
        **BASE_TRAIN_ARGS,
        **profile,
        "data": str(data_yaml_path),
        "name": profile_name,
    }


for path in [DATASET_DIR, TRAIN_PROJECT_DIR, MODEL_OUTPUT_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("=== Managed workspace ===")
print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Notebook root: {NOTEBOOKS_ROOT}")
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Dataset dir: {DATASET_DIR}")
print(f"Results dir: {TRAIN_PROJECT_DIR}")
print(f"Models dir: {MODEL_OUTPUT_DIR}")
print(f"Logs dir: {LOG_DIR}")
print(f"Training workers: {TRAIN_WORKERS}")

=== Managed workspace ===
Workspace root: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks
Notebook root: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks
Dataset dir: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\datasets\basketball-player-detection-3-14
Results dir: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\results\yolo26\basketball-player-detection-3-14
Models dir: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\models\yolo26\basketball-player-detection-3-14
Logs dir: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\logs\yolo26\basketball-player-detection-3-14
Training workers: 0


### Install dependencies for custom training


In [21]:
%pip install -q --index-url https://download.pytorch.org/whl/cu124 --extra-index-url https://pypi.org/simple torch==2.6.0+cu124 torchvision==0.21.0+cu124 torchaudio==2.6.0+cu124
%pip install -q "ultralytics>=8.4.0" supervision roboflow python-dotenv pyyaml scipy requests_toolbelt tqdm typer defusedxml

# prevent ultralytics from tracking your activity
!yolo settings sync=False
import ultralytics
ultralytics.checks()

Ultralytics 8.4.50  Python-3.13.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4090, 24563MiB)
Setup complete  (48 CPUs, 127.8 GB RAM, 1483.2/1641.6 GB disk)


## Custom Training Setup

**NOTE:** Keep the dataset under `datasets/` so Ultralytics can find it easily. Download the Roboflow export in `yolov11` format, then point training at the generated `data.yaml`.

In [22]:
print("=== Managed directories ===")
for label, directory in PIPELINE_DIRS.items():
    print(f"{label}: {directory}")
print(f"dataset: {DATASET_DIR}")
print(f"results: {TRAIN_PROJECT_DIR}")
print(f"models: {MODEL_OUTPUT_DIR}")
print(f"logs: {LOG_DIR}")

=== Managed directories ===
datasets: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\datasets
results: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\results
models: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\models
logs: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\logs
artifacts: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\artifacts
dataset: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\datasets\basketball-player-detection-3-14
results: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\results\yolo26\basketball-player-detection-3-14
models: C:\Users\patrickcruz\Documents\Professional\Github\contagem

### Dataset Bootstrap

Load the Roboflow client and download the dataset into the managed datasets directory.

In [23]:
import importlib
import os

import requests
import requests.sessions as requests_sessions
import urllib3
from dotenv import load_dotenv
from roboflow import Roboflow

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def _disable_requests_ssl_verification() -> None:
    requests_sessions_module = importlib.reload(requests_sessions)
    original_request = requests_sessions_module.Session.request

    def request_without_ssl_verification(self, method, url, **kwargs):
        kwargs.setdefault("verify", False)
        kwargs.setdefault("timeout", 30)
        return original_request(self, method, url, **kwargs)

    requests_sessions_module.Session.request = request_without_ssl_verification
    requests.Session.request = request_without_ssl_verification


def get_roboflow_client() -> Roboflow:
    _disable_requests_ssl_verification()
    load_dotenv(NOTEBOOKS_ROOT / ".env")
    api_key = os.getenv("ROBOFLOW_API_KEY")
    if not api_key:
        raise ValueError("ROBOFLOW_API_KEY is not set in notebooks/.env")
    return Roboflow(api_key=api_key)


def download_dataset(client: Roboflow, dataset_dir: Path) -> Path:
    workspace = client.workspace(ROBOFLOW_WORKSPACE)
    project = workspace.project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    dataset = version.download(EXPORT_FORMAT, location=str(dataset_dir), overwrite=True)
    return Path(dataset.location)


rf = get_roboflow_client()
print("Roboflow client ready.")

Roboflow client ready.


In [24]:
dataset_path = download_dataset(rf, DATASET_DIR)
data_yaml = dataset_path / "data.yaml"

print("=== Dataset downloaded ===")
print(f"Dataset root: {dataset_path}")
print(f"data.yaml: {data_yaml}")
print(f"Dataset exists: {dataset_path.exists()}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\datasets\basketball-player-detection-3-14 in yolo26:: 100%|██████████| 1320/1320 [00:00<00:00, 1691.65it/s]

=== Dataset downloaded ===
Dataset root: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\datasets\basketball-player-detection-3-14
data.yaml: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\datasets\basketball-player-detection-3-14\data.yaml
Dataset exists: True


### Preflight

Confirm the model weights, dataset config, and CUDA visibility before building the train args.

In [25]:
import shutil
from pathlib import Path

import yaml

workspace_root = Path.cwd().resolve()
notebooks_root = workspace_root if workspace_root.name == "notebooks" else workspace_root / "notebooks"
dataset_name = globals().get("DATASET_NAME", "basketball-player-detection-3-14")
FAST_VAL_LIMIT = 8
FAST_VALIDATION_DIR = notebooks_root / "_v"
dataset_path = globals().get("dataset_path", notebooks_root / "datasets" / dataset_name)
source_yaml = globals().get("data_yaml", dataset_path / "data.yaml")

def build_fast_validation_yaml(dataset_root: Path, source_yaml: Path, preview_dir: Path, limit: int = FAST_VAL_LIMIT) -> Path:
    config = yaml.safe_load(source_yaml.read_text(encoding="utf-8"))
    preview_images_dir = preview_dir / "images"
    preview_labels_dir = preview_dir / "labels"
    preview_images_dir.mkdir(parents=True, exist_ok=True)
    preview_labels_dir.mkdir(parents=True, exist_ok=True)

    source_val_images = dataset_root / "valid" / "images"
    source_val_labels = dataset_root / "valid" / "labels"
    if not source_val_images.exists():
        source_val_images = dataset_root / "val" / "images"
        source_val_labels = dataset_root / "val" / "labels"

    image_files = sorted(
        [path for path in source_val_images.iterdir() if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
    )
    if not image_files:
        raise RuntimeError(f"No validation images found in {source_val_images}")

    for image_path in image_files[:limit]:
        target_image = preview_images_dir / image_path.name
        shutil.copy2(image_path, target_image)
        label_path = source_val_labels / image_path.with_suffix(".txt").name
        if label_path.exists():
            shutil.copy2(label_path, preview_labels_dir / label_path.name)

    config["path"] = str(dataset_root)
    config["train"] = str(dataset_root / "train" / "images")
    config["val"] = str(preview_images_dir)

    fast_yaml_path = preview_dir / "data.fast.yaml"
    fast_yaml_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    return fast_yaml_path


training_data_yaml = build_fast_validation_yaml(dataset_path, source_yaml, FAST_VALIDATION_DIR)
print(f"Fast validation YAML: {training_data_yaml}")
print(f"Fast validation images: {len(list((FAST_VALIDATION_DIR / 'images').glob('*')))}")

Fast validation YAML: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\_v\data.fast.yaml
Fast validation images: 8


### Fast Validation Preview

Use a tiny generated validation split so Ultralytics does not spend time scanning the full validation set before training starts.

In [26]:
from pathlib import Path

import torch
from ultralytics import YOLO

def run_preflight_checks(data_yaml: Path, weights_path: Path) -> None:
    issues = []

    if not torch.cuda.is_available():
        issues.append("PyTorch cannot see a CUDA device. Install a CUDA-enabled PyTorch build and restart the kernel.")
    else:
        print(f"CUDA devices available: {torch.cuda.device_count()}")
        print(f"Active device: {torch.cuda.get_device_name(0)}")
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        print(f"CUDA memory free: {free_bytes / 1024**3:.2f} GB")
        print(f"CUDA memory total: {total_bytes / 1024**3:.2f} GB")

    if not weights_path.exists():
        issues.append(f"Model weights not found: {weights_path}")
    else:
        print(f"Model weights found: {weights_path}")

    if not data_yaml.exists():
        issues.append(f"Dataset config not found: {data_yaml}")
    else:
        print(f"Dataset config found: {data_yaml}")

    if issues:
        message = "\n".join(f"- {issue}" for issue in issues)
        raise RuntimeError(f"Preflight checks failed:\n{message}")


def load_training_model(weights_path: Path) -> YOLO:
    model = YOLO(str(weights_path))
    model.overrides["model"] = str(weights_path)
    return model


weights_path = NOTEBOOKS_ROOT / WEIGHTS_NAME
run_preflight_checks(data_yaml, weights_path)
model = load_training_model(weights_path)

CUDA devices available: 1
Active device: NVIDIA GeForce RTX 4090
CUDA memory free: 22.19 GB
CUDA memory total: 23.99 GB
Model weights found: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\yolo26n.pt
Dataset config found: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\datasets\basketball-player-detection-3-14\data.yaml


### Runtime Diagnostics

Inspect Python, CUDA, dataset, and file-system state before the train profiles are built.

In [27]:
import json
import platform
import sys
import time

import ultralytics

dataset_path = locals().get("dataset_path", DATASET_DIR)
data_yaml = locals().get("data_yaml", dataset_path / "data.yaml")
weights_path = locals().get("weights_path", NOTEBOOKS_ROOT / WEIGHTS_NAME)

print("=== Runtime diagnostics ===")
print(f"Python: {sys.version}")
print(f"Executable: {sys.executable}")
print(f"Platform: {platform.platform()}")
print(f"Torch: {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA devices: {torch.cuda.device_count()}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"CUDA memory free: {free_bytes / 1024**3:.2f} GB")
    print(f"CUDA memory total: {total_bytes / 1024**3:.2f} GB")
else:
    print("CUDA is not visible to PyTorch.")

print("=== Dataset diagnostics ===")
print(f"dataset_path: {dataset_path}")
print(f"data_yaml exists: {data_yaml.exists()}")
print(f"weights_path exists: {weights_path.exists()}")

if data_yaml.exists():
    print("--- data.yaml ---")
    print(data_yaml.read_text(encoding="utf-8", errors="ignore"))

image_files = sorted(
    [path for path in dataset_path.rglob("*") if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
 )
label_files = sorted(
    [path for path in dataset_path.rglob("*") if path.is_file() and path.suffix.lower() == ".txt"]
 )
print(f"Image files found: {len(image_files)}")
print(f"Label files found: {len(label_files)}")
print("Sample image files:")

for path in image_files[:10]:
    print(f"  {path.relative_to(dataset_path)}")
print("Sample label files:")

for path in label_files[:10]:
    print(f"  {path.relative_to(dataset_path)}")

=== Runtime diagnostics ===
Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Executable: c:\ProgramData\anaconda3\python.exe
Platform: Windows-11-10.0.26200-SP0
Torch: 2.6.0+cu124
Ultralytics: 8.4.50
CUDA available: True
CUDA devices: 1
Current device: 0
Device name: NVIDIA GeForce RTX 4090
CUDA memory free: 22.19 GB
CUDA memory total: 23.99 GB
=== Dataset diagnostics ===
dataset_path: C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\datasets\basketball-player-detection-3-14
data_yaml exists: True
weights_path exists: True
--- data.yaml ---
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 10
names: ['ball', 'ball-in-basket', 'number', 'player', 'player-in-possession', 'player-jump-shot', 'player-layup-dunk', 'player-shot-block', 'referee', 'rim']

roboflow:
  workspace: roboflow-jvuqo
  project: basketball-player-detection-3-ycjdo
  version: 14
  license: CC 

### Smoke Profile

Build the smallest valid profile first so the notebook can fail fast before the longer scale-up branch runs.

In [28]:
import json
from pathlib import Path

from ultralytics import YOLO

workspace_root = Path.cwd().resolve()
notebooks_root = workspace_root if workspace_root.name == "notebooks" else workspace_root / "notebooks"
dataset_name = globals().get("DATASET_NAME") or "basketball-player-detection-3-14"
weights_file = globals().get("WEIGHTS_NAME") or "yolo26n.pt"
dataset_dir = globals().get("DATASET_DIR") or notebooks_root / "datasets" / dataset_name
weights_path = globals().get("weights_path") or notebooks_root / weights_file
training_data_yaml = globals().get("training_data_yaml") or dataset_dir / "data.fast.yaml"
load_model = globals().get("load_training_model")

if load_model is None:
    def load_model(model_weights_path: Path) -> YOLO:
        training_model = YOLO(str(model_weights_path))
        training_model.overrides["model"] = str(model_weights_path)
        return training_model

print("=== Model diagnostics ===")
model = globals().get("model") or load_model(weights_path)
print(model)
print(f"Model class names: {model.names}")

smoke_train_args = build_train_args("smoke", training_data_yaml)
smoke_model = load_model(weights_path)

print("=== Smoke train args ===")
print(json.dumps(smoke_train_args, indent=2, default=str))

=== Model diagnostics ===
YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, a

In [29]:
smoke_start = time.perf_counter()
print("=== Smoke train start ===")
smoke_results = smoke_model.train(**smoke_train_args)
smoke_elapsed = time.perf_counter() - smoke_start
print("=== Smoke train finished ===")
print(f"Smoke elapsed: {smoke_elapsed:.2f} s")
print(smoke_results)
smoke_results

=== Smoke train start ===
New https://pypi.org/project/ultralytics/8.4.51 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.50  Python-3.13.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4090, 24563MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=1, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\_v\data.fast.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=0.01, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 2, 3, 4, 5, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000249442F4E90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047

## Training Branches

The smoke run is the active gate. The scale-up branch is optional and should only run after the smoke profile is healthy.

### Optional Scale-Up Profile

Use this only after the smoke profile passes and the dataset, GPU, and model weights look healthy.

In [30]:
import json
from pathlib import Path

from ultralytics import YOLO

workspace_root = Path.cwd().resolve()
notebooks_root = workspace_root if workspace_root.name == "notebooks" else workspace_root / "notebooks"
dataset_name = globals().get("DATASET_NAME") or "basketball-player-detection-3-14"
weights_file = globals().get("WEIGHTS_NAME") or "yolo26n.pt"
dataset_dir = globals().get("DATASET_DIR") or notebooks_root / "datasets" / dataset_name
weights_path = globals().get("weights_path") or notebooks_root / weights_file
training_data_yaml = globals().get("training_data_yaml") or dataset_dir / "data.fast.yaml"
load_model = globals().get("load_training_model")

if load_model is None:
    def load_model(model_weights_path: Path) -> YOLO:
        training_model = YOLO(str(model_weights_path))
        training_model.overrides["model"] = str(model_weights_path)
        return training_model

scale_up_train_args = build_train_args("scale_up", training_data_yaml)
scale_up_model = load_model(weights_path)

print("=== Scale-up train args ===")
print(json.dumps(scale_up_train_args, indent=2, default=str))

=== Scale-up train args ===
{
  "project": "C:\\Users\\patrickcruz\\Documents\\Professional\\Github\\contagem-de-pessoas\\count-github-yolo-01\\notebooks\\results\\yolo26\\basketball-player-detection-3-14",
  "exist_ok": true,
  "epochs": 3,
  "imgsz": 480,
  "batch": 1,
  "device": 0,
  "workers": 0,
  "cache": false,
  "pretrained": true,
  "optimizer": "AdamW",
  "lr0": 0.002,
  "cos_lr": true,
  "patience": 2,
  "fraction": 0.05,
  "mosaic": 0.25,
  "mixup": 0.0,
  "fliplr": 0.1,
  "close_mosaic": 1,
  "plots": true,
  "amp": true,
  "verbose": true,
  "save": true,
  "val": false,
  "data": "C:\\Users\\patrickcruz\\Documents\\Professional\\Github\\contagem-de-pessoas\\count-github-yolo-01\\notebooks\\_v\\data.fast.yaml",
  "name": "scale_up"
}


In [31]:
print("=== Scale-up model override check ===")
print(scale_up_model.overrides.get("model"))
print("override present:", "model" in scale_up_model.overrides)

=== Scale-up model override check ===
C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\yolo26n.pt
override present: True


### Scale-Up Training Run

Run this only after the smoke profile has passed. It keeps the same managed output layout but increases training load gradually.

In [32]:
scale_up_start = time.perf_counter()
print("=== Scale-up train start ===")
scale_up_results = scale_up_model.train(**scale_up_train_args)
scale_up_elapsed = time.perf_counter() - scale_up_start
print("=== Scale-up train finished ===")
print(f"Scale-up elapsed: {scale_up_elapsed:.2f} s")
print(scale_up_results)
scale_up_results

=== Scale-up train start ===
New https://pypi.org/project/ultralytics/8.4.51 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.50  Python-3.13.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4090, 24563MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=1, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\patrickcruz\Documents\Professional\Github\contagem-de-pessoas\count-github-yolo-01\notebooks\_v\data.fast.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.1, flipud=0.0, format=torchscript, fraction=0.05, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=480, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 2, 3, 4, 5, 7, 8, 9])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000249444A8DD0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047